In [8]:
from Bio import SeqIO
import os
from pathlib import Path
import pandas as pd

from local.constants import WORKSPACE_ROOT

In [20]:
for s in SeqIO.parse(WORKSPACE_ROOT/"data/assembly/epi300.fna", "fasta"):
    seq = str(s.seq)
    L = 5000
    head = seq[:L]
    tail = seq[-L:]
    len_seq = len(seq)

print(head)
print(tail)

TGTTTCCCGGTGAGGACGGTTACAGCCGCAGCGAGTCACTGTGGCTGGTGCGCGGCGGCGTGGCGAAACTGGATGAAGGTCACCGGCTGGCCGCACTCTGGCAGGCGCTGCCGGAAGAACTCCGCTTAAGTCCGCATCGTTATCTGGCGACAAACAGTCCGCAGGGGCCGTGGTGGCTGCTCGGTTGGTGTGAGCGGGTGCCGGAAGCGGATGAGGTGCTGCCTGCGCCGCTGCCGCCGTACCGGGTACTGACCGGGCTGGTGGACCGCTTCGGGCGCACACAGACGTTCCACCGCGAAGCCGCCGGTGAATTCAGCGGCGAAATCACCGGCGTGACGGATGGTGCCTGGCGTCACTTCCGGCTGGTACTGACCACGCAGGCGCAGCGGGCAGAAGAAGCCCGGCAGCAGGCCATTTCCGGCGGGACGGAACCGTCCGCTTTTCCTGATACCCTGCCGGGTTACACCGAATATGGCCGGGACAACGGCATCCGTCTGTCTGCCGTGTGGCTGACGCACGACCCGGAATACCCGGAGAATTTACCTGCCGCGCCGCTGGTGCGCTATGGCTGGACGCCCCGCGGCGAACTGGCGGTGGTGTATGACCGTAGTGGCAAACAGGTGCGCAGCTTTACTTACGATGATAAATACCGGGGCCGGATGGTGGCGCACCGTCACACGGGCCGGCCGGAAATCCGTTACCGTTACGACAGCGACGGGCGGGTGACAGAACAGCTAAACCCGGCAGGCTTAAGCTACACGTATCAGTATGAGAAAGACCGCATCACCATCACCGACAGCCTGGACCGCCGTGAAGTGCTGCACACGCAGGGCGAAGCCGGGCTGAAGCGGGTGGTGAAAAAGGAACACGCGGACGGCAGCGTCACGCAGAGTCAGTTTGACGCCGTGGGCAGGCTCAGGGCACAGACGGATGCCGCAGGCAGGACAACAGAGTACAGCCCGGATGTGGTGACGGGCCTCATCACGCGCATAACCACGCC

In [21]:
ws = Path("./cache/check_circularity")
ws.mkdir(exist_ok=True)
with open(ws/"q.fna", "w") as f:
    f.write(f">head"+"\n")
    f.write(f"{head}"+"\n")
    f.write(f">tail"+"\n")
    f.write(f"{tail}"+"\n")

In [22]:
COLS = "qseqid sseqid qstart qend sstart send nident qlen slen".split(" ")

os.system(f"""\
    cd cache/check_circularity
    blastn \
        -subject {WORKSPACE_ROOT}/data/asm_sean/Epi300_genome2024/Epi300_RC_100x_flye.fasta \
        -outfmt "6 {' '.join(COLS)}" \
        -query ./q.fna \
        -out vs_flye \
        
    blastn \
        -subject {WORKSPACE_ROOT}/data/assembly/epi300.fna \
        -outfmt "6 {' '.join(COLS)}" \
        -query ./q.fna \
        -out vs_self \
""")

0

In [23]:
dfs =pd.read_csv(ws/"vs_self", sep="\t", header=None)
dfs.columns = COLS
dfs

,qseqid,sseqid,qstart,qend,sstart,send,nident,qlen,slen
0,head,C1,1,5000,1,5000,5000,5000,4691561
1,head,C1,1,3313,4548739,4552051,3295,5000,4691561
2,head,C1,1,3368,1611869,1615236,3266,5000,4691561
3,head,C1,6,3338,1291515,1294880,2635,5000,4691561
4,head,C1,2319,3341,1616058,1617080,1004,5000,4691561
5,head,C1,1810,3337,2446626,2448174,1231,5000,4691561
6,head,C1,2885,3337,4639,5090,431,5000,4691561
7,head,C1,3727,4129,4553006,4553407,391,5000,4691561
8,head,C1,4639,5000,4551623,4551985,349,5000,4691561
9,head,C1,4639,5000,1616624,1616986,344,5000,4691561


In [24]:
dff =pd.read_csv(ws/"vs_flye", sep="\t", header=None)
dff.columns = COLS
dff

,qseqid,sseqid,qstart,qend,sstart,send,nident,qlen,slen
0,head,contig_1,1,5000,3121803,3126802,5000,5000,4578301
1,head,contig_1,1,3313,2978980,2982292,3295,5000,4578301
2,head,contig_1,1,3368,42110,45477,3266,5000,4578301
3,head,contig_1,6,3338,4413317,4416682,2635,5000,4578301
4,head,contig_1,2319,3341,46299,47321,1004,5000,4578301
5,head,contig_1,1810,3337,876867,878415,1231,5000,4578301
6,head,contig_1,2885,3337,3126441,3126892,431,5000,4578301
7,head,contig_1,3727,4129,2983247,2983648,391,5000,4578301
8,head,contig_1,4639,5000,2981864,2982226,349,5000,4578301
9,head,contig_1,4639,5000,46865,47227,344,5000,4578301


In [39]:
from local.figures.template import BaseFigure, ApplyTemplate, go

for df, name in [
    # (dfs, "self"),
    (dff, "flye"),
]:
    fig = BaseFigure()
    for k in ["head", "tail"]:
        _df = df[df["qseqid"]==k]
        xx, yy = [], []
        for _, r in _df.iterrows():
            qs, qe, ss, se = r[2:6]
            # ss += len_seq-L
            # se += len_seq-L
            xx += [qs, qe, None]
            yy += [ss, se, None]
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x = xx,
                y = yy,
                name=k,
            )
        )
    fig = ApplyTemplate(
        fig,
        axis={
            "1 1 y": dict(title="flye", range=[3115000, 3128000]),
            "1 1 x": dict(title="hifiasm-meta"),
        },
        layout=dict(
            title=name, width=500, height=400,
            margin=dict(l=5, r=5, t=35, b=5),
        ),
    )
    fig.show()